#### Este notebook es un clon del 06-result_movie_genre_language_managed_table de la carpeta transformation con algunas cosas modificadas para que sea carga incremental:
1. Se agrega el includes de common_functions
2. Se agrega el widget v_file_date con el nombre de las carpetas de carga incremental del contenedor bronze-inc
3. Se agrega filtro para leer las managed table particionadas de silver-inc (las que tienen datos que cambian cada semana)
4. Se modifica el mapeo de created_date, que antes mapeaba current_timestamp, ahora mapea el file_date
5. Se cambia la forma de crear la managed table en el ultimo paso, el esquema, el modo de escritura, la particion, y además se agrega un paso previo que elimine registros en caso de duplicados

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")






In [0]:
movie_df = spark.read.table("movie_silver_inc.movies") \
                     .filter(f"file_date = '{v_file_date}'")
                    





In [0]:
language_df = spark.read.table("movie_silver_inc.languages")

In [0]:
genre_df = spark.read.table("movie_silver_inc.genres")

In [0]:
movie_language_df = spark.read.table("movie_silver_inc.movies_languages") \
                              .filter(f"file_date = '{v_file_date}'")

In [0]:
movie_genre_df = spark.read.table("movie_silver_inc.movies_genres") \
                           .filter(f"file_date = '{v_file_date}'")

#### Join "languages" y "movie_languages"

In [0]:
languages_mov_lan_df = language_df.join(movie_language_df,
                                        language_df.language_id == movie_language_df.language_id,
                                        "inner") \
                                    .select(language_df.language_name, language_df.language_id, movie_language_df.movie_id)


#### Join "genres" y "movie_genres"

In [0]:
genres_mov_gen_df = genre_df.join(movie_genre_df,
                                        genre_df.genre_id == movie_genre_df.genre_id,
                                        "inner") \
                                    .select(genre_df.genre_name, genre_df.genre_id, movie_genre_df.movie_id)

#### Join "movie_df", "languages_mov_lan_df" y "genres_mov_gen_df"

In [0]:
results_movie_genre_language_df = movie_df.join(languages_mov_lan_df,
                                                movie_df.movie_id == languages_mov_lan_df.movie_id,
                                                "inner") \
                                            .join(genres_mov_gen_df,
                                                  movie_df.movie_id == genres_mov_gen_df.movie_id,
                                                  "inner") \
                                            .select(movie_df.movie_id, movie_df.title, movie_df.duration_time, movie_df.release_date, 
                                                    movie_df.vote_average, languages_mov_lan_df.language_name, languages_mov_lan_df.language_id, genres_mov_gen_df.genre_id, genres_mov_gen_df.genre_name)
                                            

In [0]:
from pyspark.sql.functions import lit






In [0]:
results_df = results_movie_genre_language_df.withColumn("created_date", lit(v_file_date))


##### Ordenar por la columna "release_date" de manera descendente

In [0]:
results_order_by_df = results_df.orderBy(results_df.release_date.desc())

In [0]:
display(results_order_by_df)

#### Escribir datos en una managed table en el contenedor gold

In [0]:
#delete_dups("movie_gold_inc","results_movie_genre_language","created_date",f"{v_file_date}")












In [0]:
#results_order_by_df.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold_inc.results_movie_genre_language")

merge_condition = 'tgt.movie_id = src.movie_id and tgt.language_id = src.language_id and tgt.genre_id = src.genre_id and tgt.created_date = src.created_date'
merge_delta_lake(results_order_by_df,'movie_gold_inc','results_movie_genre_language',merge_condition,'created_date')


















In [0]:
%sql
SELECT created_date, COUNT(1) 
FROM movie_gold_inc.results_movie_genre_language
GROUP BY created_date